In [ ]:
import subprocess
from pathlib import Path
import nbformat
from nbconvert import ScriptExporter

# Define paths
base_dir = Path.home() / "drg-pipeline" / "data-cleaning"
debug_dir = base_dir / "debug"
cache_dir = debug_dir / "cache"
converted_r_script = debug_dir / "gen-birthdate-icd-rvs-stats.r"
notebook_path = base_dir / "gen-birthdate-icd-rvs-stats.ipynb"

# Ensure cache directory exists
cache_dir.mkdir(parents=True, exist_ok=True)

# Function to convert the Jupyter notebook to an R script
def convert_notebook_to_r():
    if notebook_path.exists():
        with open(notebook_path, "r", encoding="utf-8") as nb_file:
            nb_content = nbformat.read(nb_file, as_version=4)

        # Convert only if the notebook uses R
        if nb_content.get("metadata", {}).get("kernelspec", {}).get("language", "") == "R":
            script_content, _ = ScriptExporter().from_notebook_node(nb_content)

            # Save converted script
            with open(converted_r_script, "w", encoding="utf-8") as r_file:
                r_file.write(script_content)

            print(f"Converted {notebook_path.name} to {converted_r_script}")
        else:
            print("Error: Notebook does not use R.")
    else:
        print("Error: Notebook file not found.")

# Function to run the converted R script
def run_notebook():
    print(f"Running: Rscript {converted_r_script}")  # Print command
    result = subprocess.run(["Rscript", str(converted_r_script)], capture_output=True, text=True)

    # Print both stdout and stderr for debugging
    # print("---- STDOUT ----")
    # print(result.stdout)
    # print("---- STDERR ----")
    # print(result.stderr)

    if result.returncode != 0:
        print(f"Error running {converted_r_script}. Exit code: {result.returncode}")
        return False
    return True


# Loop over the years (2018-2023)
for year in range(2018, 2024):
    year_file = cache_dir / "year.txt"
    # Write the current year to a cache file
    with open(year_file, "w", encoding="utf-8") as f:
        f.write(str(year))

    print(f"Processing year: {year}")

    # Convert the notebook before running the script
    convert_notebook_to_r()

    # Run the script and stop if it fails
    if not run_notebook():
        raise RuntimeError("R script failed.")
